## Speech Recognition & Conversational Systems

## Case Study - Option A: ASR Robustness in Noisy Environments
**Before running:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all. Takes roughly 30-60 min (MUSAN streaming is the slow part).

Fill in your details in cell 2.

In [1]:
!pip -q install transformers jiwer librosa soundfile reportlab accelerate


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
STUDENT_NAME = "G K Mahanidhi"          # <- check spelling / add surname
ROLL_NO = "23BTRCL139"# <- fill in
CODE_URL = "https://colab.research.google.com/drive/1O7IgBogOtfVlTyp-3eriI5CRvA0mamti?usp=sharing"  # <- set sharing to 'Anyone with the link can view'


## Cell 3: experiments (writes experiments.py and runs it)

In [3]:
%%writefile experiments.py
"""
Case Study - Option A: Robustness of ASR in noisy environments.

Clean speech (LibriSpeech test-clean) is mixed with MUSAN noise at 0/10/20 dB SNR.
Whisper is evaluated on the raw noisy audio (baseline) and on audio processed by
spectral subtraction (experimental). Everything is written to ./results for report.py.
"""
import os, io, sys, json, time, random, tarfile, pickle, platform, urllib.request
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal

SR = 16000

DEFAULT_CFG = dict(
    work_dir="work",
    out_dir="results",
    seed=42,
    utts_per_speaker=3,          # test-clean has 40 speakers -> 120 utterances
    min_dur=2.0, max_dur=15.0,   # seconds
    snrs=[0, 10, 20],
    noise_types=["babble", "music", "noise"],   # babble = 4 summed MUSAN speech clips (cafe chatter proxy)
    noise_files_per_type=20,
    models=["openai/whisper-tiny.en", "openai/whisper-base.en"],
    primary_model="openai/whisper-base.en",
    batch_size=16,
    ss=dict(win=400, n_fft=512, hop=160, alpha=2.0, beta=0.02, noise_pct=10),
)

LIBRISPEECH_URL = "https://www.openslr.org/resources/12/test-clean.tar.gz"
MUSAN_URL = "https://www.openslr.org/resources/17/musan.tar.gz"


def log(*a):
    print(*a, flush=True)


# ----------------------------------------------------------------- data
def _extract(tar, dest):
    try:
        tar.extractall(dest, filter="data")
    except TypeError:
        tar.extractall(dest)


def prepare_librispeech(cfg):
    root = os.path.join(cfg["work_dir"], "LibriSpeech", "test-clean")
    if not os.path.isdir(root):
        os.makedirs(cfg["work_dir"], exist_ok=True)
        tgz = os.path.join(cfg["work_dir"], "test-clean.tar.gz")
        if not os.path.exists(tgz):
            log("Downloading LibriSpeech test-clean (~346 MB) ...")
            urllib.request.urlretrieve(LIBRISPEECH_URL, tgz)
        with tarfile.open(tgz) as t:
            _extract(t, cfg["work_dir"])
    items = []
    for spk in sorted(os.listdir(root)):
        for chap in sorted(os.listdir(os.path.join(root, spk))):
            d = os.path.join(root, spk, chap)
            with open(os.path.join(d, f"{spk}-{chap}.trans.txt")) as f:
                for line in f:
                    uid, txt = line.strip().split(" ", 1)
                    p = os.path.join(d, uid + ".flac")
                    dur = sf.info(p).duration
                    if cfg["min_dur"] <= dur <= cfg["max_dur"]:
                        items.append(dict(id=uid, speaker=spk, text=txt, path=p, dur=dur))
    rng = random.Random(cfg["seed"])
    by = {}
    for it in items:
        by.setdefault(it["speaker"], []).append(it)
    sel = []
    for spk in sorted(by):
        sel += rng.sample(by[spk], min(cfg["utts_per_speaker"], len(by[spk])))
    return sel


def _to_16k(x, sr):
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        g = np.gcd(int(sr), SR)
        x = signal.resample_poly(x, SR // g, int(sr) // g)
    return x.astype(np.float32)


def load_musan(cfg):
    """Stream the MUSAN tarball and keep a few clips per category (stops early)."""
    cache = os.path.join(cfg["work_dir"], "musan_subset.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f:
            return pickle.load(f)
    need = cfg["noise_files_per_type"]
    bank = {"speech": [], "music": [], "noise": []}
    names = {"speech": [], "music": [], "noise": []}
    log("Streaming MUSAN (11 GB archive; stops as soon as enough clips are found) ...")
    try:
        resp = urllib.request.urlopen(MUSAN_URL)
        with tarfile.open(fileobj=resp, mode="r|gz") as t:
            for m in t:
                if not (m.isfile() and m.name.endswith(".wav")):
                    continue
                parts = m.name.split("/")
                cat = parts[1] if len(parts) > 2 else None
                if cat in bank and len(bank[cat]) < need:
                    x, sr = sf.read(io.BytesIO(t.extractfile(m).read()), dtype="float32")
                    x = _to_16k(x, sr)[: 60 * SR]
                    if len(x) > 2 * SR and np.mean(x ** 2) > 1e-10:
                        bank[cat].append(x)
                        names[cat].append(os.path.basename(m.name))
                if all(len(v) >= need for v in bank.values()):
                    break
    except Exception as e:  # network drop: continue if we already have something usable
        log("MUSAN stream interrupted:", repr(e))
    if any(len(v) < 3 for v in bank.values()):
        raise RuntimeError(f"Not enough MUSAN clips collected: { {k: len(v) for k, v in bank.items()} }. Re-run the cell.")
    out = dict(bank=bank, names=names)
    os.makedirs(cfg["work_dir"], exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump(out, f)
    return out


# ----------------------------------------------------------------- signal processing
def _crop(x, n, rng):
    if len(x) < n:
        x = np.tile(x, int(np.ceil(n / len(x))))
    st = int(rng.integers(0, len(x) - n + 1))
    return x[st:st + n]


def make_noise(kind, n, rng, bank):
    if kind == "babble":
        segs = [_crop(bank["speech"][int(rng.integers(len(bank["speech"])))], n, rng) for _ in range(4)]
        return sum(s / (np.sqrt(np.mean(s ** 2)) + 1e-8) for s in segs).astype(np.float32)
    return _crop(bank[kind][int(rng.integers(len(bank[kind])))], n, rng).astype(np.float32)


def mix_at_snr(s, n, snr_db):
    ps, pn = np.mean(s ** 2), np.mean(n ** 2) + 1e-12
    g = np.sqrt(ps / (pn * 10 ** (snr_db / 10)))
    y = s + g * n
    peak = np.abs(y).max()
    if peak > 0.99:
        y = y * (0.99 / peak)          # level change only; SNR is unchanged
    return y.astype(np.float32)


def measured_snr(s, y_noisy_unscaled_noise):  # helper used in sanity checks
    return 10 * np.log10(np.mean(s ** 2) / (np.mean(y_noisy_unscaled_noise ** 2) + 1e-12))


def spectral_subtraction(x, p):
    """Power spectral subtraction (Boll, 1979) with over-subtraction and a spectral floor.
    Noise PSD = mean power of the lowest-energy p['noise_pct'] % of frames."""
    ov = p["win"] - p["hop"]
    _, _, Z = signal.stft(x, fs=SR, window="hann", nperseg=p["win"], noverlap=ov, nfft=p["n_fft"])
    pw = np.abs(Z) ** 2
    e = pw.sum(0)
    k = max(3, int(len(e) * p["noise_pct"] / 100))
    npsd = pw[:, np.argsort(e)[:k]].mean(axis=1, keepdims=True)
    clean_pw = np.maximum(pw - p["alpha"] * npsd, p["beta"] * pw)
    Zc = np.sqrt(clean_pw) * np.exp(1j * np.angle(Z))
    _, y = signal.istft(Zc, fs=SR, window="hann", nperseg=p["win"], noverlap=ov, nfft=p["n_fft"])
    y = y[: len(x)]
    if len(y) < len(x):
        y = np.pad(y, (0, len(x) - len(y)))
    return y.astype(np.float32)


# ----------------------------------------------------------------- ASR + metrics
def get_normalizer(pipe):
    try:
        f = pipe.tokenizer.normalize
        f("Hello, World 20!")
        return f
    except Exception:
        import re
        return lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]", " ", s.lower())).strip()


def transcribe(pipe, wavs, bs, use_cuda):
    import torch
    inputs = [{"raw": w.astype(np.float32), "sampling_rate": SR} for w in wavs]
    if use_cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = pipe(inputs, batch_size=bs)
    if use_cuda:
        torch.cuda.synchronize()
    return [o["text"] for o in out], time.perf_counter() - t0


def utt_counts(ref, hyp):
    import jiwer
    if not ref.strip():
        return None
    if not hyp.strip():
        n, c = len(ref.split()), len(ref)
        return (0, n, 0, n, 0, c, 0, c)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return (w.substitutions, w.deletions, w.insertions, w.substitutions + w.deletions + w.hits,
            c.substitutions, c.deletions, c.insertions, c.substitutions + c.deletions + c.hits)


# ----------------------------------------------------------------- main
def run(cfg=None):
    import torch
    from transformers import pipeline
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    out_dir = cfg["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    use_cuda = torch.cuda.is_available()
    device = "cuda" if use_cuda else "cpu"
    log("Device:", device, torch.cuda.get_device_name(0) if use_cuda else "(no GPU: Runtime > Change runtime type > GPU is much faster)")

    utts = prepare_librispeech(cfg)
    clean = [_to_16k(*sf.read(u["path"], dtype="float32")) for u in utts]
    musan = load_musan(cfg)
    bank = musan["bank"]
    log(f"{len(utts)} utterances, {len({u['speaker'] for u in utts})} speakers, {sum(u['dur'] for u in utts) / 60:.1f} min")

    noises = {}
    for i in range(len(utts)):
        for k, kind in enumerate(cfg["noise_types"]):
            noises[(i, kind)] = make_noise(kind, len(clean[i]), np.random.default_rng(cfg["seed"] + 1000 * i + k), bank)

    conds = [("clean", None)] + [(n, s) for n in cfg["noise_types"] for s in cfg["snrs"]]
    wav_cache, ss_time = {}, {}

    def get_wavs(cond):
        if cond not in wav_cache:
            noise, snr = cond
            noisy = clean if noise == "clean" else [mix_at_snr(clean[i], noises[(i, noise)], snr) for i in range(len(utts))]
            t0 = time.perf_counter()
            enh = [spectral_subtraction(x, cfg["ss"]) for x in noisy]
            ss_time[cond] = time.perf_counter() - t0
            wav_cache[cond] = (noisy, enh)
        return wav_cache[cond]

    rows, timing, params = [], [], {}
    total_audio = float(sum(len(c) for c in clean)) / SR

    for model in cfg["models"]:
        log("\n=== Model:", model)
        pipe = pipeline("automatic-speech-recognition", model=model, device=0 if use_cuda else -1)
        params[model] = int(sum(p.numel() for p in pipe.model.parameters()))
        norm = get_normalizer(pipe)
        transcribe(pipe, clean[:2], 2, use_cuda)  # warm-up
        for cond in conds:
            noisy, enh = get_wavs(cond)
            for pname, wavs, extra in (("baseline", noisy, 0.0), ("spectral_sub", enh, ss_time[cond])):
                hyps, dt = transcribe(pipe, wavs, cfg["batch_size"], use_cuda)
                for u, h in zip(utts, hyps):
                    ref, hyp = norm(u["text"]), norm(h)
                    c = utt_counts(ref, hyp)
                    if c is None:
                        continue
                    rows.append(dict(model=model, pipeline=pname, noise=cond[0], snr=cond[1] if cond[1] is not None else np.nan,
                                     utt_id=u["id"], speaker=u["speaker"], ref=ref, hyp=hyp,
                                     wS=c[0], wD=c[1], wI=c[2], wN=c[3], cS=c[4], cD=c[5], cI=c[6], cN=c[7]))
                timing.append(dict(model=model, pipeline=pname, noise=cond[0], snr=cond[1] if cond[1] is not None else np.nan,
                                   asr_s=dt, ss_s=extra, audio_s=total_audio, n_utts=len(utts)))
                d = pd.DataFrame(rows)
                m = d[(d.model == model) & (d.pipeline == pname) & (d.noise == cond[0]) & (d.snr.fillna(-1) == (cond[1] if cond[1] is not None else -1))]
                log(f"{cond[0]:>6} {str(cond[1]):>4} dB {pname:>12}: WER {100 * (m.wS + m.wD + m.wI).sum() / m.wN.sum():5.1f}%  ({dt:5.1f}s)")
        del pipe
        if use_cuda:
            torch.cuda.empty_cache()

    # example for the annotated spectrogram
    idx = next((i for i, u in enumerate(utts) if 4.0 <= u["dur"] <= 8.0), 0)
    noisy, enh = (lambda c: (c[0][idx], c[1][idx]))(get_wavs(("babble", 0)))
    np.savez_compressed(os.path.join(out_dir, "spec_example.npz"), clean=clean[idx], noisy=noisy, enhanced=enh,
                        utt_id=utts[idx]["id"])

    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "per_utt.csv"), index=False)
    pd.DataFrame(timing).to_csv(os.path.join(out_dir, "timing.csv"), index=False)
    import transformers, librosa, jiwer
    meta = dict(
        n_utts=len(utts), n_speakers=len({u["speaker"] for u in utts}), total_audio_s=total_audio,
        mean_dur_s=total_audio / len(utts), min_dur=cfg["min_dur"], max_dur=cfg["max_dur"],
        musan_counts={k: len(v) for k, v in bank.items()}, params=params, device=device,
        gpu=torch.cuda.get_device_name(0) if use_cuda else "CPU", cfg=cfg,
        versions=dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                      librosa=librosa.__version__, numpy=np.__version__),
    )
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2, default=str)
    log("\nDone. Results written to", out_dir)


if __name__ == "__main__":
    run()


Writing experiments.py


In [4]:
# PATCH 1: length-capped Whisper decoding (prevents repetition loops). Run BEFORE experiments.run()
import sys, importlib
EXP = [(
'''def transcribe(pipe, wavs, bs, use_cuda):
    import torch
    inputs = [{"raw": w.astype(np.float32), "sampling_rate": SR} for w in wavs]
    if use_cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = pipe(inputs, batch_size=bs)
    if use_cuda:
        torch.cuda.synchronize()
    return [o["text"] for o in out], time.perf_counter() - t0
''',
'''TOKENS_PER_SEC_CAP = 8   # normal English speech is ~4 tokens/s, so this allows 2x headroom


def transcribe(pipe, wavs, bs, use_cuda):
    """Length-capped decoding: clips are batched by duration and each batch may emit at most
    TOKENS_PER_SEC_CAP tokens per second of its longest clip (+16). This stops Whisper's
    repetition/hallucination loops from running to the 448-token limit."""
    import torch
    order = sorted(range(len(wavs)), key=lambda i: len(wavs[i]))
    texts = [None] * len(wavs)
    if use_cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        cap = int(TOKENS_PER_SEC_CAP * max(len(wavs[i]) for i in idx) / SR) + 16
        out = pipe([{"raw": wavs[i].astype(np.float32), "sampling_rate": SR} for i in idx],
                   batch_size=len(idx), generate_kwargs={"max_new_tokens": cap})
        for i, o in zip(idx, out):
            texts[i] = o["text"]
    if use_cuda:
        torch.cuda.synchronize()
    return texts, time.perf_counter() - t0
''')]


def apply(path, edits):
    s = open(path).read()
    for old, new in edits:
        if new in s and old not in s:
            continue  # already patched
        if s.count(old) != 1:
            raise SystemExit(f"PATCH FAILED in {path}: expected 1 match, found {s.count(old)} for:\n{old[:80]}")
        s = s.replace(old, new)
    open(path, "w").write(s)
    print("patched", path)

apply('experiments.py', EXP)
if 'experiments' in sys.modules:
    importlib.reload(sys.modules['experiments'])


patched experiments.py


In [5]:
import experiments
experiments.run()

ModuleNotFoundError: No module named 'pandas'

## Cell 5: report generator

In [ ]:
%%writefile report.py
"""
Builds the case-study PDF (figures, tables, text) from the files written by experiments.py.
All numbers in the report are read from ./results; nothing is hard-coded.
"""
import os, json, datetime, difflib
from xml.sax.saxutils import escape
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.lib import colors
from reportlab.lib.utils import ImageReader
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph, Spacer, Image, Table, TableStyle,
                                PageBreak, KeepTogether, Flowable, CondPageBreak)

SR = 16000
INCH = 72
BLUE, ORANGE, GREY = "#0072B2", "#D55E00", "#7f7f7f"
NOISE_LABEL = {"babble": "Babble (cafe chatter)", "music": "Music", "noise": "Ambient noise"}


# ============================================================ data helpers
def load(out_dir):
    per = pd.read_csv(os.path.join(out_dir, "per_utt.csv"), keep_default_na=False)
    per["snr"] = pd.to_numeric(per["snr"], errors="coerce").fillna(-1).astype(int)
    tim = pd.read_csv(os.path.join(out_dir, "timing.csv"))
    tim["snr"] = tim["snr"].fillna(-1).astype(int)
    with open(os.path.join(out_dir, "meta.json")) as f:
        meta = json.load(f)
    return per, tim, meta


def aggregate(per, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for key, g in per.groupby(["model", "pipeline", "noise", "snr"]):
        e = (g.wS + g.wD + g.wI).values.astype(float)
        n = g.wN.values.astype(float)
        ce = (g.cS + g.cD + g.cI).values.astype(float)
        cn = g.cN.values.astype(float)
        idx = rng.integers(0, len(e), (n_boot, len(e)))
        boot = e[idx].sum(1) / n[idx].sum(1) * 100
        rows.append(dict(model=key[0], pipeline=key[1], noise=key[2], snr=key[3],
                         wer=e.sum() / n.sum() * 100, cer=ce.sum() / cn.sum() * 100,
                         lo=np.percentile(boot, 2.5), hi=np.percentile(boot, 97.5),
                         S=g.wS.sum(), D=g.wD.sum(), I=g.wI.sum(), N=n.sum()))
    return pd.DataFrame(rows)


def short(m):
    return m.split("/")[-1].replace("whisper-", "Whisper-")


# ============================================================ figures
def fig_pipeline(path):
    fig, ax = plt.subplots(figsize=(6.5, 3.3))
    ax.set_xlim(0, 100); ax.set_ylim(0, 52); ax.axis("off")

    def box(x, y, w, h, text, fc="#eef3f8", ec="#34495e", fs=7):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.3,rounding_size=1.2", fc=fc, ec=ec, lw=1))
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs)

    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=9, lw=1, color="#34495e"))

    # top row
    box(1, 38, 17, 11, "Clean speech\nLibriSpeech\ntest-clean", fc="#e8f5e9")
    box(23, 38, 17, 11, "Noise mixing\nMUSAN @ 0/10/20 dB\n(SNR scaling)", fc="#fff3e0")
    box(45, 38, 17, 11, "Pre-processing\nmono, 16 kHz,\nfloat32")
    arrow(18.5, 43.5, 22.5, 43.5); arrow(40.5, 43.5, 44.5, 43.5)
    # branches
    box(68, 42, 30, 8, "Baseline: noisy audio passed as-is", fc="#e3f2fd")
    box(68, 28, 30, 11, "Spectral subtraction\nSTFT (25 ms, hop 10 ms) > noise PSD\n> subtract > ISTFT", fc="#ffebee")
    arrow(62.5, 43.5, 67.5, 46); arrow(62.5, 43.5, 67.5, 34)
    # bottom row
    box(1, 6, 17, 13, "Feature extraction\n80-bin log-Mel\n25 ms win / 10 ms hop", fc="#f3e5f5")
    box(23, 6, 17, 13, "Acoustic model\nWhisper encoder\n(Transformer)", fc="#f3e5f5")
    box(45, 6, 17, 13, "Language model /\ndecoder\n(Whisper decoder)", fc="#f3e5f5")
    box(67, 6, 14, 13, "Text output\n+ normaliser", fc="#e8f5e9")
    box(85, 6, 14, 13, "Evaluation\nWER, CER,\nRTF", fc="#fffde7")
    ax.plot([83, 83], [28, 24], color="#34495e", lw=1); ax.plot([83, 83], [42, 24], color="#34495e", lw=1)
    ax.plot([83, 9.5], [24, 24], color="#34495e", lw=1); arrow(9.5, 24, 9.5, 19.6)
    arrow(18.5, 12.5, 22.5, 12.5); arrow(40.5, 12.5, 44.5, 12.5); arrow(62.5, 12.5, 66.5, 12.5); arrow(81.5, 12.5, 84.5, 12.5)
    fig.tight_layout(pad=0.2)
    fig.savefig(path, dpi=220); plt.close(fig)


def fig_wer_snr(agg, primary, path, models):
    fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.7), sharey=True)
    xs = [0, 10, 20]
    for ax, kind in zip(axes, ["babble", "music", "noise"]):
        for mi, model in enumerate(models):
            ls = "-" if model == primary else ":"
            alpha = 1.0 if model == primary else 0.7
            for pipe, col, mk in (("baseline", BLUE, "o"), ("spectral_sub", ORANGE, "s")):
                d = agg[(agg.model == model) & (agg.pipeline == pipe) & (agg.noise == kind)].sort_values("snr")
                lab = f"{short(model)} {'baseline' if pipe == 'baseline' else 'spec. sub.'}"
                ax.plot(d.snr, d.wer, ls, color=col, marker=mk, ms=4, lw=1.4, alpha=alpha, label=lab)
                if model == primary:
                    ax.fill_between(d.snr, d.lo, d.hi, color=col, alpha=0.12, lw=0)
        ax.set_title(NOISE_LABEL[kind], fontsize=8)
        ax.set_xticks(xs); ax.set_xlabel("SNR (dB)", fontsize=8)
        ax.tick_params(labelsize=7); ax.grid(alpha=0.25)
    axes[0].set_ylabel("WER (%)", fontsize=8)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=min(4, len(l)), fontsize=6.5, frameon=False)
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    fig.savefig(path, dpi=220); plt.close(fig)


def fig_spec(npz_path, path):
    import librosa
    d = np.load(npz_path)
    sigs = [d["clean"], d["noisy"], d["enhanced"]]
    mels = [librosa.feature.melspectrogram(y=x.astype(np.float32), sr=SR, n_fft=512, win_length=400, hop_length=160,
                                           n_mels=80, fmax=8000, power=2.0) for x in sigs]
    ref = mels[0].max()
    dbs = [np.clip(10 * np.log10(np.maximum(m / ref, 1e-10)), -80, 0) for m in mels]
    freqs = librosa.mel_frequencies(80, fmin=0, fmax=8000)
    hf = freqs >= 4000
    P = mels[0]; tot = P.sum(0); hfr = P[hf].sum(0) / (tot + 1e-12)
    fr = int(np.argmax(np.where(tot > np.percentile(tot, 40), hfr, -1)))   # strongest fricative-like frame
    pause = int(np.argmin(tot[: len(tot)]))                                 # quietest frame of the clean signal
    T = P.shape[1]
    sl = lambda c: slice(max(c - 2, 0), min(c + 3, T))
    hf_lvl = [10 * np.log10(m[hf][:, sl(fr)].mean() / ref) for m in mels]
    pa_lvl = [10 * np.log10(m[:, sl(pause)].mean() / ref) for m in mels]
    yk = lambda f: float(np.interp(f, freqs, np.arange(80)))
    titles = ["(a) Clean speech", "(b) Babble noise at 0 dB SNR (baseline input)", "(c) After spectral subtraction"]
    fig, axes = plt.subplots(3, 1, figsize=(6.5, 5.0), sharex=True)
    for k, (ax, db, ti) in enumerate(zip(axes, dbs, titles)):
        im = ax.imshow(db, origin="lower", aspect="auto", cmap="magma", vmin=-80, vmax=0, extent=[0, T * 0.01, 0, 80])
        ax.set_yticks([yk(f) for f in (0, 1000, 2000, 4000, 8000)]); ax.set_yticklabels(["0", "1", "2", "4", "8"], fontsize=7)
        ax.set_ylabel("Freq (kHz)", fontsize=7); ax.set_title(ti, fontsize=8, loc="left")
        tx = fr * 0.01 + (0.35 if fr * 0.01 < T * 0.01 * 0.6 else -2.3)
        ax.annotate(f"4-8 kHz frication: {hf_lvl[k]:.0f} dB", xy=(fr * 0.01, yk(6000)), xytext=(tx, yk(6600)), fontsize=6.5,
                    color="white", arrowprops=dict(arrowstyle="->", color="white", lw=0.8),
                    bbox=dict(fc="black", ec="none", alpha=0.55, pad=1.5))
        px = pause * 0.01 + (0.35 if pause * 0.01 < T * 0.01 * 0.6 else -2.3)
        ax.annotate(f"pause (all bands): {pa_lvl[k]:.0f} dB", xy=(pause * 0.01, yk(1500)), xytext=(px, yk(300)), fontsize=6.5,
                    color="white", arrowprops=dict(arrowstyle="->", color="white", lw=0.8),
                    bbox=dict(fc="black", ec="none", alpha=0.55, pad=1.5))
    axes[-1].set_xlabel("Time (s)", fontsize=8)
    cb = fig.colorbar(im, ax=axes, pad=0.015, fraction=0.03); cb.set_label("dB re. clean peak", fontsize=7); cb.ax.tick_params(labelsize=6)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)
    return dict(hf=hf_lvl, pause=pa_lvl, utt=str(d["utt_id"]), t_fric=fr * 0.01, t_pause=pause * 0.01)


# ============================================================ error analysis
def pick_failures(per, primary, kinds, n=5):
    d = per[(per.model == primary) & (per.pipeline == "baseline") & (per.snr == 0) & (per.noise.isin(kinds))].copy()
    d["werf"] = (d.wS + d.wD + d.wI) / d.wN
    pool = d[(d.wN >= 6) & (d.werf >= 0.25) & (d.werf <= 0.9)]
    if len(pool) < n:
        pool = d[d.wN >= 4]
    picked = []
    for kind in kinds:
        g = pool[pool.noise == kind].sort_values("werf", ascending=False)
        if len(g):
            picked.append(g.iloc[0])
    taken = {(r.utt_id, r.noise) for r in picked}
    for _, r in pool.sort_values("werf", ascending=False).iterrows():
        if len(picked) >= n:
            break
        if (r.utt_id, r.noise) not in taken and r.utt_id not in {p.utt_id for p in picked}:
            picked.append(r); taken.add((r.utt_id, r.noise))
    return picked[:n]


def diagnose(r):
    import jiwer
    S, D, I, N = int(r.wS), int(r.wD), int(r.wI), int(r.wN)
    clause = {"babble": "competing talkers share the spectro-temporal structure of the target speech",
              "music": "sustained harmonic music overlaps the speech formants",
              "noise": "broadband noise raises the floor over weak consonants"}[r.noise]
    try:
        o = jiwer.process_words(r.ref, r.hyp)
        refw, hypw = o.references[0], o.hypotheses[0]
        pairs = []
        for ch in o.alignments[0]:
            if ch.type == "substitute":
                for a, b in zip(refw[ch.ref_start_idx:ch.ref_end_idx], hypw[ch.hyp_start_idx:ch.hyp_end_idx]):
                    pairs.append((a, b, difflib.SequenceMatcher(None, a, b).ratio()))
    except Exception:
        pairs = []
    ex = ", ".join(f"'{a}' as '{b}'" for a, b, _ in pairs[:2])
    close = [p for p in pairs if p[2] >= 0.5]
    hyp_ratio = len(r.hyp.split()) / max(N, 1)
    if D >= S and D >= I:
        t = f"Deletion-dominant ({D} of {S + D + I} errors): weak function words and consonants are masked, since {clause}."
        if hyp_ratio < 0.6:
            t += " Hypothesis is truncated."
    elif I > S:
        t = f"Insertion-dominant ({I} insertions): the model transcribes noise as words, as {clause}."
    else:
        t = f"Substitution-dominant ({S} of {S + D + I}): "
        t += (f"mostly acoustically similar confusions ({ex}); " if len(close) >= max(1, len(pairs) // 2) and ex
              else (f"mismatches such as {ex}; " if ex else ""))
        t += f"{clause}."
    return t


# ============================================================ PDF plumbing
class Marker(Flowable):
    def __init__(self, name, store):
        super().__init__(); self.name, self.store = name, store

    def wrap(self, aw, ah):
        return (0, 0)

    def draw(self):
        self.store[self.name] = self.canv.getPageNumber()


def make_styles():
    base = dict(fontName="Times-Roman", fontSize=11, leading=12.65)
    S = {}
    S["body"] = ParagraphStyle("body", alignment=TA_JUSTIFY, spaceAfter=5, **base)
    S["bullet"] = ParagraphStyle("bullet", alignment=TA_JUSTIFY, leftIndent=16, bulletIndent=4, spaceAfter=3, **base)
    S["h1"] = ParagraphStyle("h1", fontName="Times-Bold", fontSize=13, leading=15.5, spaceBefore=8, spaceAfter=5)
    S["h2"] = ParagraphStyle("h2", fontName="Times-Bold", fontSize=11, leading=12.65, spaceBefore=4, spaceAfter=3)
    S["cap"] = ParagraphStyle("cap", fontName="Times-Italic", fontSize=10, leading=11.5, alignment=TA_CENTER, spaceBefore=2, spaceAfter=8)
    S["tcap"] = ParagraphStyle("tcap", fontName="Times-Italic", fontSize=10, leading=11.5, alignment=TA_CENTER, spaceBefore=4, spaceAfter=3)
    S["tc"] = ParagraphStyle("tc", fontName="Times-Roman", fontSize=9, leading=10.4)
    S["tcb"] = ParagraphStyle("tcb", fontName="Times-Bold", fontSize=9, leading=10.4)
    S["tcc"] = ParagraphStyle("tcc", fontName="Times-Roman", fontSize=9, leading=10.4, alignment=TA_CENTER)
    S["tcbc"] = ParagraphStyle("tcbc", fontName="Times-Bold", fontSize=9, leading=10.4, alignment=TA_CENTER)
    S["ref"] = ParagraphStyle("ref", fontName="Times-Roman", fontSize=11, leading=12.65, leftIndent=24, firstLineIndent=-24, spaceAfter=4)
    S["title"] = ParagraphStyle("title", fontName="Times-Bold", fontSize=22, leading=27, alignment=TA_CENTER, spaceAfter=18)
    S["tp"] = ParagraphStyle("tp", fontName="Times-Roman", fontSize=13, leading=20, alignment=TA_CENTER)
    S["tpb"] = ParagraphStyle("tpb", fontName="Times-Bold", fontSize=13, leading=20, alignment=TA_CENTER)
    return S


def tbl(data, widths, S, header=1, align_center_from=1, extra=None):
    rows = []
    for ri, row in enumerate(data):
        r = []
        for ci, c in enumerate(row):
            if isinstance(c, str):
                if ri < header:
                    st = S["tcbc"] if ci >= align_center_from else S["tcb"]
                else:
                    st = S["tcc"] if ci >= align_center_from else S["tc"]
                c = Paragraph(c, st)
            r.append(c)
        rows.append(r)
    t = Table(rows, colWidths=widths, repeatRows=header)
    style = [("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#888888")),
             ("BACKGROUND", (0, 0), (-1, header - 1), colors.HexColor("#e6e6e6")),
             ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
             ("TOPPADDING", (0, 0), (-1, -1), 2), ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
             ("LEFTPADDING", (0, 0), (-1, -1), 3), ("RIGHTPADDING", (0, 0), (-1, -1), 3)]
    if extra:
        style += extra
    t.setStyle(TableStyle(style))
    return t


def img(path, width_in):
    iw, ih = ImageReader(path).getSize()
    w = width_in * INCH
    return Image(path, width=w, height=w * ih / iw)


# ============================================================ report
def build_report(out_dir="results", name="Student Name", roll="ROLL_NO", code_url="https://github.com/<user>/<repo>",
                 pdf_path=None, title="Robustness of Whisper ASR in Noisy Environments: Baseline versus Spectral Subtraction"):
    per, tim, meta = load(out_dir)
    cfg = meta["cfg"]
    primary = cfg["primary_model"]
    models = [m for m in cfg["models"] if m in set(per.model)]
    kinds = cfg["noise_types"]
    snrs = cfg["snrs"]
    agg = aggregate(per)
    figdir = os.path.join(out_dir, "figures"); os.makedirs(figdir, exist_ok=True)
    p_pipe, p_wer, p_spec = (os.path.join(figdir, f) for f in ("fig_pipeline.png", "fig_wer_snr.png", "fig_spectrogram.png"))
    fig_pipeline(p_pipe)
    fig_wer_snr(agg, primary, p_wer, models)
    spec = fig_spec(os.path.join(out_dir, "spec_example.npz"), p_spec)

    def W(model, pipe, noise, snr, col="wer"):
        r = agg[(agg.model == model) & (agg.pipeline == pipe) & (agg.noise == noise) & (agg.snr == snr)]
        return float(r[col].iloc[0])

    f1 = lambda x: f"{x:.1f}"
    P = primary
    noisy_conds = [(k, s) for k in kinds for s in snrs]
    clean_b, clean_s = W(P, "baseline", "clean", -1), W(P, "spectral_sub", "clean", -1)
    avg_b = np.mean([W(P, "baseline", k, s) for k, s in noisy_conds])
    avg_s = np.mean([W(P, "spectral_sub", k, s) for k, s in noisy_conds])
    delta_rel = (avg_s - avg_b) / avg_b * 100
    better = [(k, s) for k, s in noisy_conds if W(P, "spectral_sub", k, s) < W(P, "baseline", k, s)]
    n_better, n_cond = len(better), len(noisy_conds)
    diffs = {(k, s): W(P, "spectral_sub", k, s) - W(P, "baseline", k, s) for k, s in noisy_conds}
    most_helped = min(diffs, key=diffs.get); most_hurt = max(diffs, key=diffs.get)
    worst = max(noisy_conds, key=lambda c: W(P, "baseline", *c))
    kind_delta = {k: np.mean([diffs[(k, s)] for s in snrs]) for k in kinds}
    ss_helps = delta_rel < 0
    if n_better == 0:
        verdict = f"did not lower WER in any of the {n_cond} noisy conditions"
    elif n_better == n_cond:
        verdict = f"lowered WER in all {n_cond} noisy conditions"
    else:
        verdict = f"lowered WER in {n_better} of {n_cond} noisy conditions and raised it in the other {n_cond - n_better}"
    lbl = lambda c: f"{NOISE_LABEL[c[0]].lower()} at {c[1]} dB"

    # timing / system metrics
    def sysm(model, pipe):
        t = tim[(tim.model == model) & (tim.pipeline == pipe) & (tim.noise != "clean")]
        return (t.asr_s.sum() + t.ss_s.sum()) / t.audio_s.sum(), (t.asr_s.sum() + t.ss_s.sum()) / t.n_utts.sum() * 1000
    ss_only_rtf = tim[(tim.model == P) & (tim.pipeline == "spectral_sub") & (tim.noise != "clean")]
    ss_rtf = ss_only_rtf.ss_s.sum() / ss_only_rtf.audio_s.sum()
    mp = meta["params"]
    dur_min = meta["total_audio_s"] / 60

    S = make_styles()
    store = {}
    cnt = dict(fig=0, tab=0)

    def fcap(t):
        cnt["fig"] += 1
        return Paragraph(f"<b>Figure {cnt['fig']}.</b> {t}", S["cap"])

    def tcap(t):
        cnt["tab"] += 1
        return Paragraph(f"<b>Table {cnt['tab']}.</b> {t}", S["tcap"])

    B = lambda t: Paragraph(t, S["body"])
    H1 = lambda t: KeepTogether([CondPageBreak(110), Paragraph(t, S["h1"])])
    H2 = lambda t: Paragraph(t, S["h2"])
    today = datetime.date.today().strftime("%d %B %Y")

    def story(scale):
        cnt.update(fig=0, tab=0)
        st = []
        # ------------------------------------------------ title page
        st += [Spacer(1, 70), Paragraph("JAIN (Deemed-to-be University)", S["tpb"]),
               Paragraph("School of Computer Science and Engineering (CSE - AIML)", S["tp"]), Spacer(1, 45),
               Paragraph("CASE STUDY REPORT", S["tpb"]), Spacer(1, 12), Paragraph(escape(title), S["title"]), Spacer(1, 18),
               Paragraph("<b>Case Study Option Selected:</b> Option A - Robustness in Noisy Environments (Automotive / Smart Home)", S["tp"]),
               Spacer(1, 40),
               Paragraph(f"<b>Student Name:</b> {escape(name)}", S["tp"]),
               Paragraph(f"<b>Roll Number:</b> {escape(roll)}", S["tp"]),
               Paragraph("<b>Course:</b> Speech Recognition and Conversational Systems", S["tp"]),
               Paragraph("<b>Instructor:</b> Prof. Shobha H R", S["tp"]),
               Paragraph(f"<b>Date:</b> {today}", S["tp"]), PageBreak()]

        # ------------------------------------------------ 1 executive summary
        st += [Marker("start", store), H1("1. Executive Summary")]
        st += [B(f"Voice assistants in cars and smart speakers must work where background noise is the rule rather than the exception. "
                 f"This case study measures how three kinds of background noise (babble used as a proxy for cafe chatter, music and ambient noise) "
                 f"degrade the accuracy of a modern end-to-end recogniser, and whether a classical single-channel enhancement step, spectral subtraction, "
                 f"can recover the lost accuracy. {meta['n_utts']} LibriSpeech test-clean utterances from {meta['n_speakers']} speakers "
                 f"({dur_min:.1f} minutes of audio) were mixed with MUSAN noise at 0, 10 and 20 dB SNR and transcribed by {', '.join(short(m) for m in models)}, "
                 f"with and without spectral subtraction in front of the recogniser."),
              B(f"With {short(P)}, the word error rate (WER) on clean speech is {f1(clean_b)}%. Noise raises it to between "
                f"{f1(min(W(P, 'baseline', k, s) for k, s in noisy_conds))}% and {f1(max(W(P, 'baseline', k, s) for k, s in noisy_conds))}% "
                f"depending on the condition, with the worst case being {lbl(worst)} ({f1(W(P, 'baseline', *worst))}%). "
                f"Averaged over the nine noisy conditions, the baseline WER is {f1(avg_b)}% and the spectral-subtraction pipeline gives {f1(avg_s)}% "
                f"({'a relative reduction' if ss_helps else 'a relative increase'} of {abs(delta_rel):.1f}%); spectral subtraction {verdict}."),
              B(f"<b>Conclusion.</b> " + (
                  "A lightweight, model-free enhancement front-end gives a measurable gain on this pretrained recogniser, but the size of the gain depends strongly on the noise type and SNR, so it should be applied selectively."
                  if ss_helps else
                  "A classical enhancement front-end does not help this pretrained recogniser on average: the artefacts it introduces cost as much as the noise it removes. Robustness is better pursued through noise-aware training or a learned enhancement stage than through spectral subtraction alone."))]

        # ------------------------------------------------ 2 problem statement
        st += [H1("2. Problem Statement and Real-World Context")]
        st += [B("<b>Application.</b> In-vehicle infotainment and smart-home assistants capture speech with far-field microphones while engines, road noise, "
                 "televisions, music and other people are active. The recogniser has no control over the acoustic scene, yet a misrecognised navigation "
                 "destination or device command is immediately visible to the user. Reliable performance at low signal-to-noise ratio (SNR) is therefore a "
                 "deployment requirement, not a research luxury."),
              B("<b>Why standard ASR struggles.</b> Four effects combine. (i) <i>Energetic masking</i>: noise energy in the same time-frequency cells as "
                "speech hides low-energy cues, particularly unvoiced consonants such as fricatives and stops, which carry much of the information needed to "
                "separate similar words. (ii) <i>Distribution mismatch</i>: models trained mostly on clean or moderately noisy audio see features at test time "
                "that differ from those seen in training. (iii) <i>Informational masking</i>: babble and music contain speech-like or harmonic structure, so the "
                "recogniser can attend to the wrong source, producing insertions or hallucinated words. (iv) <i>Speaker adaptation</i>: talkers raise their voice and change spectral tilt in noise "
                "(the Lombard effect), which shifts the speech away from the quiet conditions typical of read-speech corpora."),
              B("<b>Technical constraints of this study.</b>"),
              Paragraph("<bullet>&bull;</bullet><b>Single channel, no noise reference.</b> Only one microphone signal is available, so the noise spectrum has to be estimated from the signal itself.", S["bullet"]),
              Paragraph(f"<bullet>&bull;</bullet><b>Fixed SNR grid.</b> Conditions are limited to {', '.join(str(s) for s in snrs)} dB, spanning severe, moderate and mild noise.", S["bullet"]),
              Paragraph("<bullet>&bull;</bullet><b>No retraining.</b> Pretrained, publicly released checkpoints are used unchanged; the question is how well they cope as-is and whether a cheap front-end helps.", S["bullet"]),
              Paragraph("<bullet>&bull;</bullet><b>Edge-device budget.</b> Any added processing must be light enough for embedded hardware, which limits the choice of enhancement algorithm and model size.", S["bullet"]),
              Paragraph("<bullet>&bull;</bullet><b>Data realism.</b> Noise is added synthetically to read speech. This gives exact control of SNR but ignores reverberation and the Lombard effect, so absolute numbers will be optimistic relative to a real vehicle.", S["bullet"]),
              B("The investigation is guided by two questions: <b>RQ1</b> how much does each noise type degrade WER as SNR falls, and <b>RQ2</b> does spectral subtraction in front of a pretrained recogniser reduce that degradation?")]

        # ------------------------------------------------ 3 architecture
        sp = cfg["ss"]
        st += [H1("3. System Architecture and Methodology")]
        st += [B("Fig. 1 shows the evaluation pipeline. Clean utterances are mixed with noise at a controlled SNR, converted to mono 16 kHz audio, and then follow one of two "
                 "branches: the baseline passes the noisy signal directly to the recogniser, while the experimental branch first applies spectral subtraction. "
                 "Both branches use identical feature extraction, acoustic modelling and decoding, so any difference in WER is attributable to the front-end.")]
        st += [KeepTogether([img(p_pipe, 6.3 * scale), fcap("Block diagram of the evaluation pipeline (baseline and spectral-subtraction branches).")])]
        st += [H2("3.1 Signal processing and features")]
        st += [B("Speech and noise are resampled to 16 kHz mono. Noise is scaled so that the mixture has the requested SNR, "
                 "SNR<sub>dB</sub> = 10 log<sub>10</sub>(P<sub>s</sub>/P<sub>n</sub>), i.e. the noise is multiplied by "
                 "g = sqrt(P<sub>s</sub> / (P<sub>n</sub> 10<super>SNR/10</super>)). The same noise segment is reused across SNR levels for a given utterance, so the three SNR points differ only in level. "
                 "Babble is synthesised by summing four randomly chosen MUSAN speech clips, each normalised to equal power."),
              B(f"Spectral subtraction [4] operates on a Hann-windowed STFT ({sp['win']} samples = {sp['win'] / 16:.0f} ms window, {sp['hop']} samples = {sp['hop'] / 16:.0f} ms hop, {sp['n_fft']}-point FFT). "
                f"The noise power spectrum <i>N</i>(k) is estimated as the mean power of the lowest-energy {sp['noise_pct']}% of frames. Each frame is processed as "
                f"|S(k,t)|<super>2</super> = max(|Y(k,t)|<super>2</super> - <greek>a</greek>|N(k)|<super>2</super>, <greek>b</greek>|Y(k,t)|<super>2</super>), with over-subtraction factor "
                f"<greek>a</greek> = {sp['alpha']} and spectral floor <greek>b</greek> = {sp['beta']}; the noisy phase is retained and the signal is reconstructed by overlap-add. "
                f"The Whisper front-end then converts the 16 kHz signal into an 80-channel log-Mel spectrogram (25 ms window, 10 ms hop) [1]; Table 1 summarises all parameters.")]
        t1 = [["Stage", "Parameter", "Value"],
              ["Audio", "Sampling rate / channels", "16 kHz / mono"],
              ["Mixing", "SNR levels", ", ".join(f"{s} dB" for s in snrs)],
              ["Spectral subtraction", "Window / hop / FFT", f"{sp['win']} / {sp['hop']} / {sp['n_fft']} samples (Hann)"],
              ["Spectral subtraction", "Over-subtraction, floor, noise estimate", f"alpha = {sp['alpha']}, beta = {sp['beta']}, lowest {sp['noise_pct']}% frames"],
              ["Whisper feature extractor", "Representation", "80-bin log-Mel, 25 ms window, 10 ms hop"],
              ["Decoding", "Settings", "Default greedy decoding, batch size " + str(cfg["batch_size"])]]
        st += [KeepTogether([tcap("Audio and feature-extraction parameters."), tbl(t1, [105, 150, 196], S, align_center_from=9)]), Spacer(1, 6)]
        st += [H2("3.2 Model justification and configurations")]
        st += [B(f"Whisper [1] is an encoder-decoder Transformer trained on a very large, weakly supervised collection of web audio, and is widely deployed as an off-the-shelf recogniser. "
                 f"It was chosen because (a) it is representative of current end-to-end systems, (b) public checkpoints are available at several sizes, which allows an accuracy-versus-cost comparison, "
                 f"and (c) it was trained on diverse real-world audio, so it is a demanding test of whether a classical front-end still adds value. "
                 f"The English-only {short(models[0])} ({mp[models[0]] / 1e6:.0f} M parameters)" +
                 (f" and {short(models[-1])} ({mp[models[-1]] / 1e6:.0f} M parameters)" if len(models) > 1 else "") +
                 " are used so that the study can be repeated on a free GPU. Spectral subtraction is the baseline enhancement method taught in the course: it is cheap, "
                 "needs no training data and has a transparent failure mode, making it a suitable reference point for stronger methods.")]
        t2 = [["ID", "Front-end", "Recogniser", "Role"]]
        for m in models:
            t2.append([f"B-{short(m).split('-')[1]}", "None (noisy audio as-is)", short(m), "Baseline"])
        for m in models:
            t2.append([f"SS-{short(m).split('-')[1]}", "Spectral subtraction", short(m), "Experimental"])
        st += [KeepTogether([tcap("Baseline and experimental configurations."), tbl(t2, [62, 140, 140, 109], S, align_center_from=9)])]

        # ------------------------------------------------ 4 experiments
        st += [H1("4. Experimental Setup and Quantitative Results")]
        st += [H2("4.1 Dataset details")]
        mc = meta["musan_counts"]
        t3 = [["Item", "Detail"],
              ["Speech", f"LibriSpeech test-clean [2]: {meta['n_utts']} utterances, {meta['n_speakers']} speakers (equal number per speaker, random seed {cfg['seed']})"],
              ["Duration tested", f"{dur_min:.1f} min per condition (mean {meta['mean_dur_s']:.1f} s per utterance; range {meta['min_dur']:.0f}-{meta['max_dur']:.0f} s)"],
              ["Noise", f"MUSAN [3]: {mc['speech']} speech clips (summed into babble), {mc['music']} music clips, {mc['noise']} ambient-noise clips"],
              ["Conditions", f"1 clean + {len(kinds)} noise types x {len(snrs)} SNRs = {1 + len(kinds) * len(snrs)} conditions, each run with {2} front-ends and {len(models)} models"],
              ["Train / test split", "Evaluation only. No model is trained or fine-tuned; the whole selected set is test data."]]
        st += [KeepTogether([tcap("Dataset summary."), tbl(t3, [85, 366], S, align_center_from=9)]), Spacer(1, 4)]
        st += [H2("4.2 Evaluation metrics")]
        st += [B("Word error rate, WER = (S + D + I) / N, and character error rate (CER, the same ratio at character level) are computed with the jiwer library over all utterances of a condition, "
                 "after applying the Whisper English text normaliser to both reference and hypothesis so that casing, punctuation and number formatting are not counted as errors. "
                 "The 95% intervals in Fig. 2 are bootstrap percentile intervals over utterances (1000 resamples). System cost is reported as real-time factor (RTF = processing time / audio duration, "
                 f"including spectral subtraction where used), mean latency per utterance, parameter count and FP32 weight size. Timings were measured on {escape(str(meta['gpu']))} with batched inference, "
                 "so they reflect throughput rather than single-request latency.")]
        st += [H2("4.3 Results")]
        rows = [["Condition", "Baseline WER (%)", "Spec. sub. WER (%)", "Baseline CER (%)", "Spec. sub. CER (%)", "Change in WER"]]
        extra = []

        def addrow(label, noise, snr):
            bw, sw = W(P, "baseline", noise, snr), W(P, "spectral_sub", noise, snr)
            rows.append([label, f1(bw), f1(sw), f1(W(P, "baseline", noise, snr, "cer")), f1(W(P, "spectral_sub", noise, snr, "cer")),
                         ("%+.1f%%" % ((sw - bw) / bw * 100)) if bw > 0 else "n/a"])
        addrow("Clean", "clean", -1)
        for k in kinds:
            for s in snrs:
                addrow(f"{NOISE_LABEL[k]}, {s} dB", k, s)
        st += [KeepTogether([tcap(f"WER and CER of {short(P)} per condition; the last column is the relative change in WER from spectral subtraction (negative = better)."),
                             tbl(rows, [120, 62, 66, 62, 66, 75], S)]), Spacer(1, 6)]
        trend = []
        for k in kinds:
            trend.append(f"{NOISE_LABEL[k].lower()} ({f1(W(P, 'baseline', k, 20))}% at 20 dB to {f1(W(P, 'baseline', k, 0))}% at 0 dB)")
        st += [B(f"<b>Effect of noise (RQ1).</b> Table 4 and Fig. 2 show that WER rises monotonically as SNR falls for the baseline: " + "; ".join(trend) +
                 f". The worst baseline condition is {lbl(worst)}. Clean-speech WER of {f1(clean_b)}% sets the floor against which these degradations should be read.")]
        st += [KeepTogether([img(p_wer, 6.4 * scale), fcap(f"WER versus SNR for each noise type. Solid lines: {short(P)}"
                                                           + (f"; dotted lines: {short([m for m in models if m != P][0])}" if len(models) > 1 else "") +
                                                           ". Blue: baseline; orange: spectral subtraction. Shaded bands: 95% bootstrap interval for the primary model.")])]
        st += [B(f"<b>Effect of spectral subtraction (RQ2).</b> Averaged over the nine noisy conditions, spectral subtraction changes the WER of {short(P)} from {f1(avg_b)}% to {f1(avg_s)}% "
                 f"({delta_rel:+.1f}% relative), and {verdict}. The largest gain is for {lbl(most_helped)} ({diffs[most_helped]:+.1f} points) and the largest loss is for {lbl(most_hurt)} ({diffs[most_hurt]:+.1f} points). "
                 f"By noise type the mean change in WER is " + ", ".join(f"{kind_delta[k]:+.1f} points for {NOISE_LABEL[k].lower()}" for k in kinds) + ". "
                 f"On clean speech the front-end changes WER from {f1(clean_b)}% to {f1(clean_s)}%, which quantifies the cost of the processing itself. "
                 + ("A plausible reading is that the noise estimate from low-energy frames is reasonable for stationary noise but poor for babble and music, whose spectra change continually; "
                    "over-subtraction then removes speech energy and leaves isolated spectral peaks ('musical noise') that the recogniser has not seen in training. "
                    "This interpretation is consistent with the per-noise pattern above but was not tested directly here.")),
              ]
        t5 = [["Model", "Params (M)", "FP32 size (MB)", "Clean WER (%)", "Noisy WER (%) baseline", "Noisy WER (%) spec. sub.", "RTF baseline", "RTF spec. sub.", "ms / utt (baseline)"]]
        for m in models:
            ab = np.mean([W(m, "baseline", k, s) for k, s in noisy_conds]); a2 = np.mean([W(m, "spectral_sub", k, s) for k, s in noisy_conds])
            rb, lb = sysm(m, "baseline"); rs, _ = sysm(m, "spectral_sub")
            t5.append([short(m), f"{mp[m] / 1e6:.1f}", f"{mp[m] * 4 / 1e6:.0f}", f1(W(m, "baseline", "clean", -1)), f1(ab), f1(a2), f"{rb:.4f}", f"{rs:.4f}", f"{lb:.1f}"])
        st += [KeepTogether([tcap("Model comparison. Noisy WER is the mean over the nine noisy conditions; RTF for spectral subtraction includes the front-end."),
                             tbl(t5, [60, 38, 44, 40, 52, 56, 42, 46, 73], S)]), Spacer(1, 6)]
        sc = f"{spec['hf'][0]:.0f} dB to {spec['hf'][1]:.0f} dB"
        st += [B(f"<b>Acoustic evidence.</b> Fig. 3 shows log-Mel spectrograms of utterance {escape(spec['utt'])}. In the clean signal the strongest high-frequency (4-8 kHz) frame, near t = {spec['t_fric']:.2f} s and typical of a fricative, "
                 f"sits at {spec['hf'][0]:.0f} dB relative to the clean peak. With babble at 0 dB the same region measures {spec['hf'][1]:.0f} dB, and the quietest frame of the clean signal "
                 f"(a pause near t = {spec['t_pause']:.2f} s) rises from {spec['pause'][0]:.0f} dB to {spec['pause'][1]:.0f} dB: the noise fills the gaps between words and covers weak consonant cues. "
                 f"After spectral subtraction these two measurements become {spec['hf'][2]:.0f} dB and {spec['pause'][2]:.0f} dB respectively.")]
        st += [KeepTogether([img(p_spec, 6.0 * scale), fcap("Annotated log-Mel spectrograms of one test utterance: clean, babble at 0 dB SNR, and after spectral subtraction. Annotations give the level of a high-frequency (fricative-like) frame and of a pause.")])]

        # ------------------------------------------------ 5 error analysis
        st += [H1("5. Qualitative Error Analysis")]
        fails = pick_failures(per, P, kinds)
        st += [B(f"Five utterances were selected from the {short(P)} baseline at 0 dB SNR: the most severe partial failure for each noise type, followed by the next-worst remaining cases. "
                 f"Utterances that were almost completely wrong were skipped because they offer little diagnostic information. Table 6 gives the ground truth and the prediction, and the cause attributed to each error "
                 f"based on the word-level alignment (substitutions, deletions, insertions) and the acoustic properties of the interfering signal.")]
        t6 = [["#", "Condition", "Ground truth", "Model prediction", "Diagnosis"]]
        for i, r in enumerate(fails, 1):
            t6.append([str(i), f"{NOISE_LABEL[r.noise].split(' (')[0]}, 0 dB<br/>{escape(r.utt_id)}<br/>WER {(r.wS + r.wD + r.wI) / r.wN * 100:.0f}%",
                       escape(r.ref), escape(r.hyp if r.hyp.strip() else "(empty)"), escape(diagnose(r))])
        st += [KeepTogether([tcap("Ground truth versus prediction for five failure cases (normalised text)."),
                             tbl(t6, [16, 62, 118, 118, 137], S, align_center_from=9)]), Spacer(1, 6)]
        nS, nD, nI = (sum(int(getattr(r, c)) for r in fails) for c in ("wS", "wD", "wI"))
        st += [B(f"Across these five samples there are {nS} substitutions, {nD} deletions and {nI} insertions. "
                 + ("Deletions dominate, which fits energetic masking of short, weak words. " if nD >= max(nS, nI) else
                    "Substitutions dominate, which fits partial masking in which the cue is present but ambiguous. " if nS >= nI else
                    "Insertions are the largest category, which fits informational masking in which interfering sound is decoded as speech. ")
                 + "Errors on babble tend to involve words that were never spoken by the target talker, whereas errors on ambient noise are more often dropped or "
                   "acoustically similar words. These attributions come from the alignment pattern and the nature of each interferer; confirming them for an individual sample requires listening to the audio.")]

        # ------------------------------------------------ 6 trade-offs
        small, big = models[0], models[-1]
        rb_s, lb_s = sysm(small, "baseline"); rb_b, lb_b = sysm(big, "baseline")
        ab_s = np.mean([W(small, "baseline", k, s) for k, s in noisy_conds]); ab_b = np.mean([W(big, "baseline", k, s) for k, s in noisy_conds])
        st += [H1("6. Engineering Trade-offs and Conclusion")]
        st += [H2("6.1 Trade-offs observed")]
        if small != big:
            st += [B(f"<b>Accuracy versus cost.</b> Moving from {short(small)} ({mp[small] / 1e6:.0f} M parameters, {mp[small] * 4 / 1e6:.0f} MB in FP32) to {short(big)} ({mp[big] / 1e6:.0f} M, {mp[big] * 4 / 1e6:.0f} MB) "
                     f"changes the mean noisy-condition baseline WER from {f1(ab_s)}% to {f1(ab_b)}% while the real-time factor goes from {rb_s:.4f} to {rb_b:.4f} on {escape(str(meta['gpu']))}. "
                     f"The larger model is therefore {'more accurate' if ab_b < ab_s else 'not more accurate'} here at a cost of roughly {rb_b / max(rb_s, 1e-9):.1f}x the processing time and {mp[big] / mp[small]:.1f}x the memory, "
                     f"which matters for edge devices with a few hundred megabytes of RAM and no GPU.")]
        st += [B(f"<b>Front-end cost versus benefit.</b> Spectral subtraction runs at an RTF of about {ss_rtf:.4f} on a CPU-side NumPy/SciPy implementation, negligible next to the neural recogniser, and needs no training data. "
                 f"Its benefit, however, is {'positive but uneven' if ss_helps else 'not positive on average'} ({delta_rel:+.1f}% relative WER), and it slightly "
                 f"{'raises' if clean_s > clean_b else 'does not raise'} the clean-speech error ({f1(clean_b)}% to {f1(clean_s)}%). A system that always applies it pays a small penalty when conditions are good, "
                 f"so enhancement should be gated on an estimated SNR or noise-stationarity measure."),
              B(f"<b>Evaluation caveats.</b> Results use synthetic mixing of read speech, {meta['n_utts']} utterances per condition and one random noise draw per utterance; intervals in Fig. 2 capture utterance sampling variability but not the choice of noise clips. "
                f"Real vehicles add reverberation, moving sources and the Lombard effect.")]
        st += [H2("6.2 Recommendations for future work")]
        r1 = ("Spectral subtraction gives little or negative benefit, so effort is better spent on the recogniser itself. " if not ss_helps else
              "Because the front-end gain is limited and noise dependent, the recogniser itself should also be made robust. ")
        st += [Paragraph(f"<bullet>1.</bullet><b>Train or fine-tune with noise augmentation.</b> {r1}Fine-tune the chosen Whisper size on speech mixed with MUSAN-style noise over a 0-20 dB SNR range (and add reverberation), "
                         f"and measure WER on held-out noise clips to confirm the gain generalises.", S["bullet"]),
                 Paragraph(f"<bullet>2.</bullet><b>Replace or gate the enhancement stage.</b> Compare spectral subtraction with a learned single-channel enhancer and with a noise-adaptive estimator (for example minimum-statistics tracking), "
                           f"and switch it on only when the estimated SNR is low and the noise is stationary. The per-noise differences in Section 4 ({', '.join(f'{NOISE_LABEL[k].split(chr(32))[0].lower()} {kind_delta[k]:+.1f}' for k in kinds)} WER points on average) indicate where gating would matter most.", S["bullet"]),
                 Paragraph("<bullet>3.</bullet><b>Optimise for the target device.</b> Evaluate int8 quantisation or distillation of the selected model and re-measure WER, RTF and memory on the actual embedded CPU, "
                           "adding a voice-activity detector so the recogniser and enhancer run only on speech segments.", S["bullet"])]
        st += [B(f"<b>Conclusion.</b> Background noise is a major source of recognition errors for a pretrained end-to-end recogniser, with WER rising from {f1(clean_b)}% on clean speech to as much as {f1(W(P, 'baseline', *worst))}% in the worst condition. "
                 + ("Classical spectral subtraction recovers part of this loss but not uniformly." if ss_helps else "Classical spectral subtraction does not recover this loss on average.")
                 + " Reliable in-vehicle and smart-home deployment will need noise-aware training and selective enhancement rather than a single fixed front-end.")]
        st += [Marker("end", store), PageBreak()]

        # ------------------------------------------------ references + appendix
        refs = [
            'A. Radford, J. W. Kim, T. Xu, G. Brockman, C. McLeavey, and I. Sutskever, "Robust speech recognition via large-scale weak supervision," arXiv:2212.04356, 2022.',
            'V. Panayotov, G. Chen, D. Povey, and S. Khudanpur, "LibriSpeech: An ASR corpus based on public domain audio books," in <i>Proc. IEEE ICASSP</i>, 2015, pp. 5206-5210.',
            'D. Snyder, G. Chen, and D. Povey, "MUSAN: A music, speech, and noise corpus," arXiv:1510.08484, 2015.',
            'S. F. Boll, "Suppression of acoustic noise in speech using spectral subtraction," <i>IEEE Trans. Acoust., Speech, Signal Process.</i>, vol. 27, no. 2, pp. 113-120, Apr. 1979.',
            'T. Wolf <i>et al.</i>, "Transformers: State-of-the-art natural language processing," in <i>Proc. EMNLP: System Demonstrations</i>, 2020, pp. 38-45.',
            'A. Paszke <i>et al.</i>, "PyTorch: An imperative style, high-performance deep learning library," in <i>Advances in Neural Information Processing Systems 32</i>, 2019.',
            'B. McFee <i>et al.</i>, "librosa: Audio and music signal analysis in Python," in <i>Proc. 14th Python in Science Conf.</i>, 2015, pp. 18-25.',
            'J. D. Hunter, "Matplotlib: A 2D graphics environment," <i>Comput. Sci. Eng.</i>, vol. 9, no. 3, pp. 90-95, 2007.',
            'N. Vaessen, "jiwer: Similarity measures for automatic speech recognition evaluation," Python package, https://github.com/jitsi/jiwer.',
        ]
        st += [Paragraph("References", S["h1"])] + [Paragraph(f"[{i}] {r}", S["ref"]) for i, r in enumerate(refs, 1)]
        st += [Spacer(1, 10), Paragraph("Appendix A: Reproducibility", S["h1"]),
               Paragraph(f"<b>Code repository / Colab notebook:</b> {escape(code_url)}", S["body"]),
               Paragraph("<b>Software versions:</b> " + ", ".join(f"{k} {v}" for k, v in meta["versions"].items()) + f". Hardware: {escape(str(meta['gpu']))}. Random seed: {cfg['seed']}.", S["body"]),
               Paragraph(f"<b>Experiment configuration:</b> {escape(json.dumps({k: v for k, v in cfg.items() if k not in ('work_dir', 'out_dir')}))}", S["body"]),
               Paragraph("Appendix B: Full results", S["h1"])]
        ta = [["Model", "Front-end", "Condition", "WER (%)", "95% CI", "CER (%)"]]
        for m in models:
            for pipe in ("baseline", "spectral_sub"):
                for noise, snr in [("clean", -1)] + noisy_conds:
                    r = agg[(agg.model == m) & (agg.pipeline == pipe) & (agg.noise == noise) & (agg.snr == snr)].iloc[0]
                    cond = "Clean" if noise == "clean" else f"{NOISE_LABEL[noise].split(' (')[0]}, {snr} dB"
                    ta.append([short(m), "Baseline" if pipe == "baseline" else "Spec. sub.", cond, f1(r.wer), f"{r.lo:.1f}-{r.hi:.1f}", f1(r.cer)])
        st += [tbl(ta, [80, 70, 130, 55, 70, 46], S, align_center_from=3)]
        return st

    def on_later(canv, doc):
        canv.saveState(); canv.setFont("Times-Roman", 10)
        canv.drawCentredString(A4[0] / 2, 0.5 * INCH, f"Page {doc.page}")
        canv.restoreState()

    pdf_path = pdf_path or os.path.join(out_dir, f"{roll}_{name.replace(' ', '_')}_CaseStudy.pdf")
    for scale in (1.0, 0.92, 0.85, 0.78):
        store.clear()
        doc = BaseDocTemplate(pdf_path, pagesize=A4, leftMargin=INCH, rightMargin=INCH, topMargin=INCH, bottomMargin=INCH,
                              title=title, author=name)
        frame = Frame(INCH, INCH, A4[0] - 2 * INCH, A4[1] - 2 * INCH, leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0)
        doc.addPageTemplates([PageTemplate(id="p", frames=[frame], onPage=lambda c, d: (on_later(c, d) if d.page > 1 else None))])
        doc.build(story(scale))
        body_pages = store["end"] - store["start"] + 1
        print(f"[layout] figure scale {scale}: body pages = {body_pages} (limit 6-10)")
        if body_pages <= 10:
            break
    if not 6 <= body_pages <= 10:
        print("WARNING: body length is outside 6-10 pages; adjust text/figure sizes.")
    return pdf_path, body_pages


if __name__ == "__main__":
    print(build_report())


In [ ]:
# PATCH 2: report layout fixes + decoding note. Run BEFORE 'import report'
import sys, importlib
REP = [
('''    S["bullet"] = ParagraphStyle("bullet",''',
 '''    S["bodyx"] = ParagraphStyle("bodyx", alignment=TA_JUSTIFY, spaceAfter=5, fontName="Times-Roman", fontSize=11, leading=15)
    S["left"] = ParagraphStyle("left", alignment=TA_LEFT, spaceAfter=5, **base)
    S["bullet"] = ParagraphStyle("bullet",'''),
('''    H2 = lambda t: Paragraph(t, S["h2"])''',
 '''    H2 = lambda t: KeepTogether([CondPageBreak(70), Paragraph(t, S["h2"])])
    BX = lambda t: Paragraph(t, S["bodyx"])'''),
('''B("Speech and noise are resampled''', '''BX("Speech and noise are resampled'''),
('''B(f"Spectral subtraction [4] operates''', '''BX(f"Spectral subtraction [4] operates'''),
('''"Default greedy decoding, batch size " + str(cfg["batch_size"])]]''',
 '''"Greedy decoding, batch size " + str(cfg["batch_size"]) + ", output capped at 8 tokens per second of audio + 16"]]'''),
('''"so they reflect throughput rather than single-request latency.")]''',
 '''"so they reflect throughput rather than single-request latency. Decoding is length-capped (at most 8 tokens per second of audio plus 16, about twice the normal speaking rate) "
                 "so that Whisper's well-known repetition loops on very noisy input cannot run to the maximum output length and dominate the averages.")]'''),
('''tbl(t5, [60, 38, 44, 40, 52, 56, 42, 46, 73], S)''', '''tbl(t5, [74, 36, 40, 38, 50, 54, 40, 44, 68], S)'''),
('''Paragraph(f"<b>Code repository / Colab notebook:</b> {escape(code_url)}", S["body"])''',
 '''Paragraph(f"<b>Code repository / Colab notebook:</b> {escape(code_url)}", S["left"])'''),
('''Random seed: {cfg['seed']}.", S["body"])''', '''Random seed: {cfg['seed']}.", S["left"])'''),
('''if k not in ('work_dir', 'out_dir')}))}", S["body"])''', '''if k not in ('work_dir', 'out_dir')}))}", S["left"])'''),
]


def apply(path, edits):
    s = open(path).read()
    for old, new in edits:
        if new in s and old not in s:
            continue  # already patched
        if s.count(old) != 1:
            raise SystemExit(f"PATCH FAILED in {path}: expected 1 match, found {s.count(old)} for:\n{old[:80]}")
        s = s.replace(old, new)
    open(path, "w").write(s)
    print("patched", path)

apply('report.py', REP)
if 'report' in sys.modules:
    importlib.reload(sys.modules['report'])

In [ ]:
import report
pdf, pages = report.build_report('results', name=STUDENT_NAME, roll=ROLL_NO, code_url=CODE_URL)
print(pdf, 'body pages:', pages)

In [ ]:
from google.colab import files
files.download(pdf)